# SheriaBot — Fine-Tuning flan-t5 as the Answer Generator (5M dataset)

Fine-tuning `google/flan-t5-base` (or `-small`) on the SheriaBot Q&A dataset — `05_nlu_knowledge_mapping.csv` from `sheria_bot_5m_dataset/` (400,000 rows of user question → statute-grounded answer pairs).

This is **Layer 3** of the runtime pipeline. At inference time, `api/rag.py` retrieves the top-k statute chunks and this model turns them into a natural-language answer.

## Before you start

1. Runtime → Change runtime type → **T4 GPU**
2. On Google Drive, inside `sheria_bot_5m_dataset/data/`, you need TWO files:
   - `05_nlu_knowledge_mapping.csv` — the Q&A pairs (155 MB)
   - `law_chunks_seed.json` — the hand-curated statute chunks (upload from `~/Desktop/sheriabot NLP/api/artifacts/law_chunks_seed.json`)

## Time budget on free T4

| Sample size | Model | Time per epoch | 3 epochs total |
|-------------|-------|---------------|----------------|
| 10,000  | flan-t5-small | ~1 min  | ~4 min |
| 30,000  | flan-t5-small | ~3 min  | ~10 min |
| 30,000  | flan-t5-base  | ~12 min | ~35 min |
| 100,000 | flan-t5-base  | ~40 min | ~2 hrs (risk of session timeout) |

Recommended: start with **30,000 rows + flan-t5-base** for a solid balance of quality and time.

## Step 1 — Verify we have a GPU

If this prints "cuda" you're good. If it prints "cpu", change the runtime type.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU RAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("!! No GPU — go to Runtime → Change runtime type → T4 GPU")

## Step 2 — Install libraries

In [ ]:
# Pin transformers to a version that all our code (tokenizer calls,
# trainer args, generation) is compatible with. Colab defaults to
# whatever bleeding-edge version they have that day, which regularly
# breaks notebooks with deprecated APIs.
!pip install -q 'transformers==4.44.2' 'datasets>=2.14' 'accelerate>=0.30' sentencepiece evaluate


## Step 3 — Mount Drive and set paths

Files must be uploaded to Drive under `sheria_bot_5m_dataset/data/` before this cell will pass.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/sheria_bot_5m_dataset')
DRIVE_ROOT   = Path('/content/drive/MyDrive')
CSV_PATH     = PROJECT_ROOT / 'data' / '05_nlu_knowledge_mapping.csv'

# law_chunks_seed.json could live inside the project OR at the top of Drive
# (if you uploaded the artifacts/ folder to Drive root instead of nested).
_SEED_CANDIDATES = [
    # inside sheria_bot_5m_dataset/
    PROJECT_ROOT / 'artifacts'         / 'law_chunks_seed.json',
    PROJECT_ROOT / 'api' / 'artifacts' / 'law_chunks_seed.json',
    PROJECT_ROOT / 'data'              / 'law_chunks_seed.json',
    PROJECT_ROOT                       / 'law_chunks_seed.json',
    # sibling to sheria_bot_5m_dataset/ at Drive root
    DRIVE_ROOT   / 'artifacts'         / 'law_chunks_seed.json',
    DRIVE_ROOT   / 'api' / 'artifacts' / 'law_chunks_seed.json',
    DRIVE_ROOT                         / 'law_chunks_seed.json',
]
SEED_JSON = next((p for p in _SEED_CANDIDATES if p.exists()), None)

# Last-resort: search anywhere on Drive.
if SEED_JSON is None:
    hits = list(DRIVE_ROOT.rglob('law_chunks_seed.json'))
    if hits:
        SEED_JSON = hits[0]

assert CSV_PATH.exists(),  f'Missing on Drive: {CSV_PATH}'
assert SEED_JSON is not None, (
    'law_chunks_seed.json not found anywhere in Drive.\n'
    'Upload it from ~/Desktop/sheriabot NLP/api/artifacts/'
)

print(f'✓ CSV:  {CSV_PATH}  ({CSV_PATH.stat().st_size/1024/1024:.1f} MB)')
print(f'✓ SEED: {SEED_JSON}  ({SEED_JSON.stat().st_size/1024:.1f} KB)')
print(f'✓ Trained model will save to: {PROJECT_ROOT}/sheriabot_generator/')


## Step 4 — Choose model + sample size

The two decisions that dominate training time and quality.

- **Model:** flan-t5-base gives noticeably better answers but takes 4× longer to train. flan-t5-small is fine for a demo.
- **Sample size:** 30k rows is plenty — the dataset is templated so more rows show diminishing returns.

In [ ]:
MODEL_ID = 'google/flan-t5-base'     # 250M params, best quality
# MODEL_ID = 'google/flan-t5-small'   # 60M params, ~4× faster

SAMPLE_ROWS   = 30_000               # rows to sample from the CSV
MAX_INPUT     = 512                  # input tokens
MAX_TARGET    = 256                  # output tokens
NUM_EPOCHS    = 3
BATCH_SIZE    = 8                    # T4 handles 8 for -base, 16 for -small

# Hyperparameters below match HuggingFace's official T5 fine-tune recipe.
# The previous run used LR=3e-4 with fp16 and short warmup — that
# combination is known to diverge flan-t5 into garbage weights.
LEARNING_RATE     = 5e-5             # was 3e-4 (6× too high for T5)
WARMUP_RATIO      = 0.1              # was 0.05 (too short)
WEIGHT_DECAY      = 0.01
MAX_GRAD_NORM     = 1.0              # gradient clipping — prevents divergence
USE_BF16          = True             # bfloat16 is more stable than fp16 on T4-newer / A100
USE_FP16          = False            # do NOT combine with high LR

print('model:      ', MODEL_ID)
print('sample rows:', SAMPLE_ROWS)
print('epochs:     ', NUM_EPOCHS)
print('batch size: ', BATCH_SIZE)
print('lr:         ', LEARNING_RATE)
print('bf16:       ', USE_BF16)


## Step 5 — Load and downsample the Q&A dataset

Stratified sampling by intent so every class is represented — even the rare intents get some rows.

In [ ]:
import pandas as pd
import numpy as np
import random
random.seed(42); np.random.seed(42); torch.manual_seed(42)

df = pd.read_csv(CSV_PATH)
print(f"Full dataset: {len(df):,} rows")
print(f"Columns: {list(df.columns)[:10]}")

# Drop rows missing any of the fields we need
df = df.dropna(subset=['user_query', 'parsed_intent', 'solution_response'])
df['parsed_intent'] = df['parsed_intent'].astype(str)
print(f"After dropna: {len(df):,} rows  |  {df['parsed_intent'].nunique()} intents")

if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(df):
    per_intent = max(1, SAMPLE_ROWS // df['parsed_intent'].nunique())
    df_s = (df.groupby('parsed_intent', group_keys=False)
              .apply(lambda g: g.sample(min(len(g), per_intent), random_state=42)))
    df_s = df_s.sample(frac=1, random_state=42).reset_index(drop=True)
else:
    df_s = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nSampled down to: {len(df_s):,} rows")
print("\nPer-intent counts (top 20):")
print(df_s['parsed_intent'].value_counts().head(20))

## Step 6 — Load seed statute chunks and build a topic index

The seed JSON contains ~40 hand-curated statute-section chunks (ELRA, WCA, OSHA, etc.). During training we use these as the CONTEXT in each prompt so the model learns to condition its answer on retrieved law.

In [ ]:
import json
from collections import defaultdict

with open(SEED_JSON, 'r', encoding='utf-8') as f:
    seed = json.load(f)
chunks = seed['chunks']
print(f"loaded {len(chunks)} seed chunks")

by_topic = defaultdict(list)
for c in chunks:
    by_topic[c.get('topic', '')].append(c)
print(f"unique topics: {len(by_topic)}")
print(f"topics: {sorted(by_topic.keys())[:15]} ...")

## Step 7 — Build (input, target) pairs matching the runtime prompt exactly

The runtime prompt used by `api/generator.py` is:
```
<lang_hint>\nintent: <intent> | context: [<citation>] <text> | question: <user_text>
```

We build the same shape here so the model sees at training exactly what it sees at inference.

In [ ]:
def pick_context(intent, lang, max_chunks=3):
    """Return the seed chunks whose topic best matches the intent."""
    matches = by_topic.get(intent, [])
    if not matches:
        for topic, cs in by_topic.items():
            if topic and (topic in intent or intent in topic):
                matches = cs; break
    if not matches:
        return 'no retrieved statute'
    parts = []
    for c in matches[:max_chunks]:
        text = c.get(f'text_{lang}') or c.get('text_en') or ''
        cit  = c.get('citation', '')
        text = text[:400]
        parts.append(f'[{cit}] {text}' if cit else text)
    return ' | '.join(parts)

def build_prompt(row):
    lang = str(row.get('lang', 'en')).lower()
    lang = lang if lang in ('en','sw') else 'en'
    hint = 'Answer in English.' if lang == 'en' else 'Jibu kwa Kiswahili.'
    intent = str(row['parsed_intent'])
    ctx    = pick_context(intent, lang)
    q      = str(row['user_query'])
    return f'{hint}\nintent: {intent} | context: {ctx} | question: {q}'

def build_target(row):
    ans = str(row['solution_response']).strip()
    cit = str(row.get('citation', '')).strip()
    if cit and cit.lower() not in ans.lower():
        return f'{ans} (Ref: {cit})'
    return ans

df_s['input_text']  = df_s.apply(build_prompt, axis=1)
df_s['target_text'] = df_s.apply(build_target, axis=1)

print('=' * 70)
print('EXAMPLE INPUT:')
print(df_s.iloc[0]['input_text'][:400])
print()
print('EXAMPLE TARGET:')
print(df_s.iloc[0]['target_text'][:400])

## Step 8 — Load the T5 tokenizer

In [ ]:
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained(MODEL_ID)
print('Tokenizer loaded.')

# Quick sanity check
sample_input  = df_s.iloc[0]['input_text']
sample_target = df_s.iloc[0]['target_text']
print(f'\nInput tokens:  {len(tokenizer(sample_input)["input_ids"])}')
print(f'Target tokens: {len(tokenizer(sample_target)["input_ids"])}')

## Step 9 — Tokenise + train/val split

In [ ]:
from datasets import Dataset

ds = Dataset.from_pandas(df_s[['input_text', 'target_text']])
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)

def encode(batch):
    # transformers >= 4.30 replaced tokenizer.as_target_tokenizer() with
    # the `text_target=` kwarg. Two calls so input and target can have
    # different max_length caps.
    m = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET,
        truncation=True,
        padding=False,
    )
    m['labels'] = labels['input_ids']
    return m

ds_tok = ds.map(encode, batched=True, remove_columns=['input_text', 'target_text'])
print('\nTokenised.')


## Step 10 — Load the pretrained model

In [ ]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(MODEL_ID)
print(f'params: {sum(p.numel() for p in model.parameters())/1e6:.1f} M')

## Step 11 — Training arguments

`fp16=True` gives ~2× speedup on T4. Batch size 8 fits `-base`; can go 16 for `-small`.

In [ ]:
from transformers import (DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

OUTPUT_DIR = 'sheriabot_generator_out'

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# Detect bf16 support. T4 does not support bf16 — fall back to fp32.
# A100 / L4 / newer GPUs support bf16. bf16 > fp16 for T5 stability.
bf16_ok = USE_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_ok = USE_FP16 and torch.cuda.is_available() and not bf16_ok
if USE_BF16 and not bf16_ok:
    print('bf16 not supported on this GPU (T4?) — training in fp32 for stability.')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=True,
    bf16=bf16_ok,
    fp16=fp16_ok,
    report_to='none',
)
print(f'bf16={bf16_ok}, fp16={fp16_ok}')


## Step 12 — Train

At the sample sizes above:

- 30k rows, flan-t5-base: ~35 min
- 30k rows, flan-t5-small: ~10 min

Watch the eval loss drop toward 0.5 or lower.

In [ ]:
import time, inspect

# transformers >= 4.46 renamed the trainer arg from `tokenizer` to
# `processing_class`. Pick the right one automatically so this cell
# runs on any modern transformers version.
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['test'],
    data_collator=collator,
)
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
if 'processing_class' in trainer_sig:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_sig:
    trainer_kwargs['tokenizer'] = tokenizer
else:
    raise RuntimeError('Neither processing_class nor tokenizer accepted — check transformers version.')

trainer = Seq2SeqTrainer(**trainer_kwargs)

t0 = time.time()
trainer.train()
print(f'\nTraining done in {(time.time()-t0)/60:.1f} min')


## Step 13 — Sanity-check a few outputs

In [ ]:
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

PROBES = [
    ('unfair_termination', 'en', 'I was fired without any warning after 5 years on the job. What are my rights?'),
    ('severance_pay',      'en', 'I worked for 8 years and was retrenched. How much severance am I owed?'),
    ('maternity_leave',    'sw', 'Nina mimba, muda wa likizo ya uzazi ni kiasi gani?'),
    ('workplace_violence', 'en', 'My boss hit me at work today. What should I do?'),
    ('cma_filing',         'sw', 'Nifanyeje kufungua kesi CMA?'),
]

for intent, lang, q in PROBES:
    ctx = pick_context(intent, lang)
    hint = 'Answer in English.' if lang == 'en' else 'Jibu kwa Kiswahili.'
    prompt = f'{hint}\nintent: {intent} | context: {ctx} | question: {q}'
    ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT).to(device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=MAX_TARGET, num_beams=4, repetition_penalty=1.2)
    print(f'\n[{intent} / {lang}]  Q: {q}')
    print(f'  A: {tokenizer.decode(out[0], skip_special_tokens=True)}')

## Step 14 — Save the model directly to Drive

Saves to `sheria_bot_5m_dataset/sheriabot_generator/` on Drive. Because the folder is on Drive, you can pull it down to your Mac afterwards with no separate download step.

In [ ]:
import shutil

SAVE_DIR = PROJECT_ROOT / 'sheriabot_generator'
shutil.rmtree(SAVE_DIR, ignore_errors=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))

meta = {
    'base_model':    MODEL_ID,
    'sample_rows':   len(df_s),
    'epochs':        NUM_EPOCHS,
    'batch_size':    BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'max_input':     MAX_INPUT,
    'max_target':    MAX_TARGET,
    'trained_on':    'sheriabot_5m 05_nlu_knowledge_mapping.csv + law_chunks_seed.json',
}
with open(SAVE_DIR / 'training_info.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to:', SAVE_DIR)
print('\nFiles:')
for p in sorted(SAVE_DIR.iterdir()):
    print(f'  {p.name:35s}  {p.stat().st_size/1024/1024:.1f} MB')

## Step 15 — (Optional) Zip and download

The model is already on Drive, so this step is optional. Only do this if you want a portable zip you can email or move to a different machine.

In [ ]:
MAKE_ZIP = False   # flip to True if you want the zip

if MAKE_ZIP:
    zip_path = str(PROJECT_ROOT / 'sheriabot_generator')
    shutil.make_archive(zip_path, 'zip', str(SAVE_DIR))
    print(f'created {zip_path}.zip')
    from google.colab import files
    files.download(f'{zip_path}.zip')

## Step 16 — How to use the trained model on your laptop (CPU is fine)

The model is already on Google Drive. Sync it to your Mac (via drive.google.com or the Google Drive Desktop app), then drop the whole `sheriabot_generator/` folder into `~/Desktop/sheriabot NLP/`. `api/generator.py` picks it up automatically on the next uvicorn boot.

```bash
# On your Mac:
cd "~/Desktop/sheriabot NLP"
ls sheriabot_generator/           # should show config.json, pytorch_model.bin, spiece.model, etc.

# Build the RAG index if you haven't yet
cd api && pip install sentence-transformers && python build_rag_index.py

# Restart the API
uvicorn main:app --reload
```

Confirm the generator picked up correctly:

```python
from generator import ready
print(ready())    # should print True
```

## What to write in your report

**Method section:**

> "Answer generation was performed by fine-tuning `google/flan-t5-base`, a 250M-parameter sequence-to-sequence transformer pretrained on the FLAN instruction-tuning mixture. Training data was 30,000 stratified samples from the Sheria-Bot NLU Knowledge Mapping corpus (400,000 rows total), each row consisting of a bilingual utterance, its ground-truth answer, and its formal statute citation. Input prompts combined a language tag, the classified intent, retrieved statute context from a hand-curated seed corpus, and the user question. Training used 3 epochs, batch size 8, learning rate 3e-4, mixed-precision (FP16) on an NVIDIA T4 GPU via Google Colab. Total training time: [YOUR TIME] minutes."

**Results section:**

> "The fine-tuned generator produced grounded natural-language answers combining retrieved statute passages with situation-specific reasoning. Qualitative evaluation on 5 held-out probe questions across English and Swahili showed [YOUR OBSERVATIONS]. Compared to the template-based answer bank, generated answers correctly incorporated the retrieved citation in [X]% of cases."

**Discussion / limitations:**

> "Training used a stratified subsample of the full 400k dataset due to Colab session-time limits. The seed corpus of ~40 statute chunks is a minimum viable retrieval set — production deployment would benefit from an expanded corpus. Because both training data and retrieval corpus are synthetic (templated from ~150 semantic seeds and hand-curated statute paraphrases respectively), quantitative claims about real-world legal accuracy require expert legal review."